# V1.5 Feature Engineering for 15-Minute Data

This notebook follows the same structure as `feature_engineering.ipynb`
but adapted for 15-minute resolution data.

- Lag features use 15-min steps (lag_1 = 15min ago, lag_96 = 24h ago, lag_672 = 7d ago)
- Rolling windows: 4 = 1h, 24 = 6h, 96 = 24h, 672 = 7d

In [ ]:
import pandas as pd
import numpy as np
import holidays

## 1. Load Data

In [ ]:
df = pd.read_csv("V1.5_15min_Dataset.csv", parse_dates=["datetime"])
df = df.rename(columns={
    "temperature_c": "temp",
    "wind_speed_ms": "wind_speed",
})
df = df.sort_values("datetime").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["datetime"].min(), "→", df["datetime"].max())
df.head(3)

## 2. Temporal Features

Basic calendar components extracted from the datetime index.
`time_of_day` maps each 15-min slot to 0-95 (96 slots per day).

In [ ]:
df["hour"]         = df["datetime"].dt.hour
df["minute"]       = df["datetime"].dt.minute
df["day_of_week"]  = df["datetime"].dt.dayofweek   # 0=Mon, 6=Sun
df["day_of_month"] = df["datetime"].dt.day
df["month"]        = df["datetime"].dt.month
df["week_of_year"] = df["datetime"].dt.isocalendar().week.astype(int)
df["quarter"]      = df["datetime"].dt.quarter
df["year"]         = df["datetime"].dt.year

# time_of_day: 96 unique slots per day (0=00:00, 1=00:15, ..., 95=23:45)
df["time_of_day"] = df["hour"] * 4 + df["minute"] // 15

season_map = {12: 1, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2,
              6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4}
df["season"] = df["month"].map(season_map)

df["is_weekend"]    = (df["day_of_week"] >= 5).astype(int)
df["is_peak_hour"]  = (df["hour"].isin(range(7, 10)) | df["hour"].isin(range(17, 21))).astype(int)
df["is_night_hour"] = (df["hour"].isin(range(23, 24)) | df["hour"].isin(range(0, 6))).astype(int)

df[["datetime","hour","minute","time_of_day","day_of_week","month","season","is_weekend","is_peak_hour"]]

## 3. Cyclic Encoding

Tree models don't know that hour 23 and hour 0 are adjacent. Sine/cosine transforms fix this.

In [ ]:
def cyclic_encode(series, max_val):
    angle = 2 * np.pi * series / max_val
    return np.sin(angle), np.cos(angle)

df["hour_sin"],        df["hour_cos"]        = cyclic_encode(df["hour"], 24)
df["day_of_week_sin"], df["day_of_week_cos"] = cyclic_encode(df["day_of_week"], 7)
df["month_sin"],       df["month_cos"]       = cyclic_encode(df["month"], 12)
df["week_of_year_sin"],df["week_of_year_cos"]= cyclic_encode(df["week_of_year"], 52)

df[["datetime","hour","hour_sin","hour_cos","month","month_sin","month_cos"]].head(6)

## 4. Finnish Public Holiday Flag

Holidays cause demand drops similar to weekends. Uses the holidays library for official Finnish holidays.

In [ ]:
years = df["datetime"].dt.year.unique().tolist()
fi_holidays = holidays.Finland(years=years)

df["is_holiday"] = df["datetime"].dt.date.astype(str).isin(
    [str(d) for d in fi_holidays.keys()]
).astype(int)

# Combined flag: any non-working hour (weekend or holiday)
df["is_non_working"] = ((df["is_weekend"] == 1) | (df["is_holiday"] == 1)).astype(int)

holiday_counts = df.groupby("is_holiday")["datetime"].count()
print("Non-holiday rows:", holiday_counts.get(0, 0))
print("Holiday rows:    ", holiday_counts.get(1, 0))
df[df["is_holiday"] == 1][["datetime","is_holiday","is_non_working"]].head(8)

## 5. Lag Features (adapted for 15-min resolution)

Lag features give the model access to historical prices.
lag_1 = 15 min ago, lag_4 = 1 hour ago, lag_96 = 24 hours ago, lag_672 = 7 days ago.

In [ ]:
# 15-min lags: 1 step = 15 minutes, 4 steps = 1 hour, 96 steps = 24 hours, 672 steps = 7 days
for steps in [1, 2, 4, 8, 16, 32, 96, 672]:
    df[f"price_lag_{steps}"] = df["price"].shift(steps)

df[["datetime", "price", "price_lag_1", "price_lag_4", "price_lag_96", "price_lag_672"]].head(10)

## 6. Rolling Features (adapted for 15-min resolution)

Capture recent price trends and volatility.
Window sizes: 4 = 1 hour, 24 = 6 hours, 96 = 24 hours, 672 = 7 days.

In [ ]:
# 1-hour rolling stats (window=4 for 15-min data)
df["price_rolling_mean_1h"] = df["price"].shift(1).rolling(4).mean()
df["price_rolling_std_1h"]  = df["price"].shift(1).rolling(4).std()

# 6-hour rolling stats (window=24)
df["price_rolling_mean_6h"] = df["price"].shift(1).rolling(24).mean()

# 24-hour rolling stats (window=96)
df["price_rolling_mean_24h"] = df["price"].shift(1).rolling(96).mean()
df["price_rolling_std_24h"]  = df["price"].shift(1).rolling(96).std()
df["price_rolling_min_24h"]  = df["price"].shift(1).rolling(96).min()
df["price_rolling_max_24h"]  = df["price"].shift(1).rolling(96).max()

# 7-day rolling mean (window=672)
df["price_rolling_mean_7d"] = df["price"].shift(1).rolling(672).mean()

# 1-hour rolling mean on temperature
df["temp_rolling_mean_1h"] = df["temp"].shift(1).rolling(4).mean()

df[["datetime","price","price_rolling_mean_1h","price_rolling_std_24h","price_rolling_mean_7d"]].head(30).tail(6)

## 7. Weather-Derived Features

Proxy features derived from temperature and wind speed that more directly reflect energy supply and demand.

In [ ]:
# Heating Degree Hours: how much colder than comfort threshold (17°C)
# Higher value = more heating demand = higher electricity price in Finland
df["HDD"] = (17 - df["temp"]).clip(lower=0)

# Wind power proxy: turbine output scales with wind³, capped at rated speed (13 m/s)
df["wind_power_proxy"] = df["wind_speed"].clip(upper=13) ** 3

# Lagged temperature (1 hour and 24 hours)
df["temp_lag_4"]  = df["temp"].shift(4)    # 1 hour ago
df["temp_lag_96"] = df["temp"].shift(96)   # 24 hours ago

df[["datetime","temp","wind_speed","HDD","wind_power_proxy","temp_lag_4"]].head(8)

## 8. Save

In [ ]:
print(f"Total columns: {len(df.columns)}")
print(df.columns.tolist())
print(f"\nNaN counts per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

df.to_csv("V1.5_15min_features.csv", index=False)
print("\nSaved → V1.5_15min_features.csv")